In [0]:
# Databricks notebook source
# Studio 5 - Profilo orario della produzione solare
#
# Qui imposto:
# - librerie;
# - catalogo;
# - tabella Gold finale;
# - controllo iniziale sui dati;
# - funzioni comuni per lettura dati, grafici e conversioni.

# Importo le librerie principali.
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots


# Imposto il catalogo e la tabella Gold finale.
catalogo = "progetto_meteo"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Controllo che la Gold finale sia popolata.
# Se la tabella è vuota, i casi studio non possono essere eseguiti.
righe_gold = spark.table(tabella_dati_studio).count()

if righe_gold == 0:
    raise Exception(
        f"La tabella {tabella_dati_studio} è vuota. "
        "Prima esegui Gold_Dati_Aggregati e Gold_Dati_Studio."
    )

print(f"Tabella pronta: {tabella_dati_studio}")
print(f"Righe disponibili: {righe_gold}")


# Leggo una query direttamente da Databricks.
# Dentro Databricks il notebook può usare Spark senza credenziali locali.
def leggi_databricks(query):

    return spark.sql(query).toPandas()


# Mostro i grafici Plotly dentro Databricks.
# displayHTML rende il grafico più stabile nei notebook Databricks.
def mostra_grafico(fig):

    displayHTML(
        fig.to_html(
            include_plotlyjs=True,
            full_html=False,
            config={"responsive": True}
        )
    )


# Converto colonne pandas in numerico quando serve.
# Questo evita problemi di visualizzazione con Plotly.
def converti_colonne_numeriche(df, colonne):

    for colonna in colonne:
        if colonna in df.columns:
            df[colonna] = pd.to_numeric(df[colonna], errors="coerce")

    return df

In [0]:
# Studio 5 - Profilo orario della produzione solare
#
# Questo studio analizza come si distribuisce la produzione solare
# durante le ore della giornata.
#
# L'analisi viene fatta per:
# - macroarea;
# - mese;
# - ora del giorno.
#
# Per ogni macroarea viene creata una heatmap in cui:
# - le righe rappresentano i mesi;
# - le colonne rappresentano le ore;
# - il colore rappresenta il valore medio della produzione solare.
#
# L'obiettivo è osservare in quali ore e in quali mesi
# la produzione solare diventa più rilevante.
#
# Questa analisi è utile perché il solare ha un profilo giornaliero molto marcato:
# tende a concentrarsi nelle ore centrali della giornata
# e cambia intensità in base alla stagione e alla macroarea.
#
# In ambito gestionale può aiutare a leggere meglio le fasce orarie
# in cui il solare contribuisce di più alla produzione rinnovabile.
#
# In ambito economico o di investimento può essere utile per valutare
# differenze territoriali, stagionalità della fonte solare e finestre orarie
# in cui la produzione media risulta più interessante.
#
# Nota importante:
# Ho voluto impostare una soglia di 300 MW, ovvero un valore considerato pari a zero durante la raccolta dei dati
# Sotto questa soglia il colore resta blu scuro
# così le ore con produzione molto bassa sono distinguibili dalle ore realmente produttive.


# Imposto l'anno da analizzare.
anno_studio = 2025


# Leggo da Databricks il solare medio per area, mese e ora.
query_profilo_solare = f"""
WITH dati_base AS (
    SELECT
        Area,
        to_date(Data, 'dd/MM/yyyy') AS Data_Convertita,
        Ora,
        Solare,
        Temperatura
    FROM {tabella_dati_studio}
    WHERE Solare IS NOT NULL
      AND Temperatura IS NOT NULL
)

SELECT
    Area,
    date_format(Data_Convertita, 'yyyy-MM') AS Mese,
    Ora,
    CAST(ROUND(AVG(Solare), 2) AS DOUBLE) AS Solare_Medio,
    CAST(ROUND(AVG(Temperatura), 2) AS DOUBLE) AS Temperatura_Media
FROM dati_base
WHERE year(Data_Convertita) = {anno_studio}
GROUP BY
    Area,
    date_format(Data_Convertita, 'yyyy-MM'),
    Ora
ORDER BY
    Area,
    Mese,
    Ora
"""

df_profilo_solare = leggi_databricks(query_profilo_solare)


# Converto le colonne numeriche.
df_profilo_solare = converti_colonne_numeriche(
    df_profilo_solare,
    [
        "Ora",
        "Solare_Medio",
        "Temperatura_Media"
    ]
)


# Controllo i dati letti.
display(df_profilo_solare)

print("Righe lette:", len(df_profilo_solare))


# Imposto l'ordine delle aree.
aree = [
    "Nord",
    "Centro",
    "Sud",
    "Isole"
]


# Imposto la soglia sotto la quale la produzione resta evidenziata come bassa.
# Questa soglia era presente nella versione originale e va mantenuta.
soglia_bassa = 300


# Calcolo il valore massimo per rendere confrontabili le quattro heatmap.
valore_massimo = max(df_profilo_solare["Solare_Medio"].max(), soglia_bassa)


# Calcolo il punto della scala colori corrispondente alla soglia.
punto_soglia = soglia_bassa / valore_massimo


# Creo la scala colori con soglia.
# Fino a 300 MW il colore resta blu scuro.
# Dopo la soglia passa alla scala giallo, arancio e rosso.
scala_solare = [
    [0.00, "#08306b"],
    [punto_soglia, "#08306b"],
    [min(punto_soglia + 0.001, 1.00), "#ffffcc"],
    [0.45, "#ffeda0"],
    [0.60, "#feb24c"],
    [0.75, "#f03b20"],
    [1.00, "#bd0026"]
]


# Creo il grafico con 4 riquadri, uno per macroarea.
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=aree,
    shared_xaxes=True,
    shared_yaxes=True
)


# Creo una heatmap per ogni area.
for indice, area in enumerate(aree):

    riga = indice // 2 + 1
    colonna = indice % 2 + 1

    df_area = df_profilo_solare[df_profilo_solare["Area"] == area]

    tabella_solare = df_area.pivot(
        index="Mese",
        columns="Ora",
        values="Solare_Medio"
    )

    tabella_temperatura = df_area.pivot(
        index="Mese",
        columns="Ora",
        values="Temperatura_Media"
    )

    testo_hover = []

    for mese in tabella_solare.index:

        riga_hover = []

        for ora in tabella_solare.columns:

            solare = tabella_solare.loc[mese, ora]
            temperatura = tabella_temperatura.loc[mese, ora]

            riga_hover.append(
                f"Area: {area}<br>"
                f"Mese: {mese}<br>"
                f"Ora: {ora}:00<br>"
                f"Solare medio: {solare:.2f} MW<br>"
                f"Temperatura media: {temperatura:.2f} °C"
            )

        testo_hover.append(riga_hover)

    fig.add_trace(
        go.Heatmap(
            z=tabella_solare.values,
            x=tabella_solare.columns,
            y=tabella_solare.index,
            text=testo_hover,
            hoverinfo="text",
            zmin=0,
            zmax=valore_massimo,
            colorscale=scala_solare,
            colorbar=dict(title="MW medi") if indice == 3 else None,
            showscale=(indice == 3)
        ),
        row=riga,
        col=colonna
    )


# Sistemo il layout finale.
fig.update_layout(
    title=f"Profilo orario della produzione solare per macroarea - {anno_studio}",
    title_x=0.5,
    height=850
)

fig.update_xaxes(
    title_text="Ora del giorno",
    tickmode="linear",
    dtick=2
)

fig.update_yaxes(
    title_text="Mese",
    type="category"
)


# Mostro il grafico.
mostra_grafico(fig)